# Passo 0 – Preparação do ambiente

Este notebook prepara tudo antes do experimento:

1. Conecta o Google Drive, onde ficam as bases e os resultados.
2. Cria as pastas do projeto.
3. Fixa a semente aleatória (42), para que os resultados possam ser repetidos.
4. Baixa os recursos do NLTK (stopwords e RSLP) e testa se funcionam.
5. Registra as versões das bibliotecas em `resultados/ambiente.txt`.

Execute as células na ordem, de cima para baixo.

## 1. Conectar o Google Drive

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
    NO_COLAB = True
except ImportError:
    # Permite rodar fora do Colab (no computador, por exemplo)
    PASTA_TCC = os.path.abspath('TCC')
    NO_COLAB = False

print('Rodando no Colab:', NO_COLAB)
print('Pasta do projeto:', PASTA_TCC)

## 2. Criar as pastas do projeto

In [ ]:
PASTAS = {
    'brutos': os.path.join(PASTA_TCC, 'dados', 'brutos'),           # bases como foram baixadas
    'processados': os.path.join(PASTA_TCC, 'dados', 'processados'), # bases limpas e unificadas
    'resultados': os.path.join(PASTA_TCC, 'resultados'),            # tabelas, gráficos e registros
}

for nome, caminho in PASTAS.items():
    os.makedirs(caminho, exist_ok=True)
    print(f'{nome:12s} -> {caminho}')

## 3. Fixar a semente aleatória

Todo sorteio do experimento (divisão dos dados, validação cruzada, XGBoost) usa esta mesma semente.

In [ ]:
import random
import numpy as np

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
os.environ['PYTHONHASHSEED'] = str(SEMENTE)

print('Semente fixada em', SEMENTE)

## 4. Baixar e testar os recursos do NLTK

- **stopwords**: lista de palavras muito frequentes em português.
- **rslp**: algoritmo que reduz as palavras ao radical (Orengo; Huyck, 2001).

In [ ]:
import nltk
for recurso in ['stopwords', 'rslp']:
    if not nltk.download(recurso, quiet=True):
        raise RuntimeError(f'Não foi possível baixar o recurso {recurso} do NLTK. Verifique a conexão.')

from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

# Termos de negação que NÃO serão removidos (decisão descrita na metodologia)
NEGACOES = {'não', 'nem', 'nunca', 'jamais', 'nenhum', 'nenhuma', 'nada'}

stop_pt = set(stopwords.words('portuguese'))
stop_final = stop_pt - NEGACOES

print('Stopwords do NLTK:', len(stop_pt))
print('Stopwords após manter as negações:', len(stop_final))
print('Negações que estavam na lista e foram mantidas:', sorted(stop_pt & NEGACOES))

radical = RSLPStemmer()
for palavra in ['eleições', 'eleitoral', 'eleitores', 'candidato', 'candidatos', 'urnas']:
    print(f'{palavra:12s} -> {radical.stem(palavra)}')

## 5. Registrar as versões das bibliotecas

As versões reais usadas aqui são as que entram no texto do TCC.

In [ ]:
import sys, platform, datetime
import pandas, sklearn, xgboost, scipy, statsmodels, matplotlib, seaborn

versoes = {
    'Python': platform.python_version(),
    'pandas': pandas.__version__,
    'NumPy': np.__version__,
    'NLTK': nltk.__version__,
    'scikit-learn': sklearn.__version__,
    'XGBoost': xgboost.__version__,
    'SciPy': scipy.__version__,
    'statsmodels': statsmodels.__version__,
    'Matplotlib': matplotlib.__version__,
    'Seaborn': seaborn.__version__,
}

linhas = [f'Registro gerado em {datetime.datetime.now():%Y-%m-%d %H:%M}', f'Semente: {SEMENTE}', '']
linhas += [f'{nome:14s} {versao}' for nome, versao in versoes.items()]
texto = '\n'.join(linhas)
print(texto)

with open(os.path.join(PASTAS['resultados'], 'ambiente.txt'), 'w', encoding='utf-8') as f:
    f.write(texto + '\n')
print('\nSalvo em resultados/ambiente.txt')

## Próximo passo

Se todas as células rodaram sem erro, o ambiente está pronto.

- Anote as versões impressas acima. Elas substituem as do texto atual da metodologia.
- O próximo notebook é o `01_bases.ipynb` (Passo 1: reunir as bases).